In [1]:
import pandas as pd
import statsmodels.api as sm

# Load the dataset
file_path = '/Users/baselhussein/Projects/impossible_goals/agg/data/merged_approaches.csv'
df = pd.read_csv(file_path)

# Define function to create binary indicators for sequences
def create_sequence_indicators(df, sequences):
    for index, row in sequences.iterrows():
        sequence_label = f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}"
        df[sequence_label] = (
            (df['lvl1'] == row['lvl1']) & 
            (df['lvl2'] == row['lvl2']) & 
            (df['lvl3'] == row['lvl3'])
        ).astype(int)
    return df

# Remove outliers (3 SD from mean)
mean_levels_complete = df['len_levels_complete'].mean()
std_levels_complete = df['len_levels_complete'].std()
lower_threshold = mean_levels_complete - 3 * std_levels_complete
upper_threshold = mean_levels_complete + 3 * std_levels_complete

data_no_outliers = df[(df['len_levels_complete'] >= lower_threshold) & (df['len_levels_complete'] <= upper_threshold)]

# Extract unique sequences of behaviors
sequences_unique = df[['lvl1', 'lvl2', 'lvl3']].drop_duplicates().reset_index(drop=True)

# Create binary indicators for each unique sequence
data_no_outliers = create_sequence_indicators(data_no_outliers, sequences_unique)

# Prepare the predictors (X) and the response variable (y) without outliers
X_no_outliers = data_no_outliers[sequences_unique.apply(lambda row: f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}", axis=1)]
y_no_outliers = data_no_outliers['len_levels_complete']

# Grand Mean Centering
X_no_outliers_centered = X_no_outliers.apply(lambda x: x - x.mean(), axis=0)

# Add a constant term to the model for the intercept
X_no_outliers_with_intercept = sm.add_constant(X_no_outliers_centered)

# Fit the linear regression model using statsmodels without outliers
model_no_outliers = sm.OLS(y_no_outliers, X_no_outliers_with_intercept)
results_no_outliers = model_no_outliers.fit()

# Extract coefficients, p-values, and confidence intervals for the model without outliers
no_outliers_summary = results_no_outliers.summary2().tables[1]

# Format p-values to three decimal places for readability
no_outliers_summary['P>|t|'] = no_outliers_summary['P>|t|'].apply(lambda p: f"{p:.3f}")

# Mark significance levels
no_outliers_summary['Significance'] = no_outliers_summary['P>|t|'].astype(float).apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
)

# Calculate the number of observations (N) for each unique sequence without outliers
sequence_counts_no_outliers = data_no_outliers.groupby(['lvl1', 'lvl2', 'lvl3']).size().reset_index(name='N')
sequence_counts_no_outliers['Sequence'] = sequence_counts_no_outliers.apply(lambda row: f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}", axis=1)
sequence_counts_no_outliers = sequence_counts_no_outliers[['Sequence', 'N']]

# Merge the sequence counts with the regression summary
sequence_labels = sequences_unique.apply(lambda row: f"{row['lvl1']}->{row['lvl2']}->{row['lvl3']}", axis=1)
no_outliers_summary['Sequence'] = ['const'] + sequence_labels.tolist()
merged_summary = no_outliers_summary.reset_index().merge(sequence_counts_no_outliers, on='Sequence', how='left')

# Sort by the number of observations (N) in descending order
merged_summary_sorted = merged_summary.sort_values(by='N', ascending=False)

# Model metrics
r_squared = results_no_outliers.rsquared
adj_r_squared = results_no_outliers.rsquared_adj
aic = results_no_outliers.aic
bic = results_no_outliers.bic

# Write results to a .txt file
with open('/Users/baselhussein/Projects/impossible_goals/agg/output/model_summary.txt', 'w') as f:
    f.write('Regression Results\n')
    f.write('=================\n\n')
    f.write(f'R-squared: {r_squared:.4f}\n')
    f.write(f'Adjusted R-squared: {adj_r_squared:.4f}\n')
    f.write(f'AIC: {aic:.4f}\n')
    f.write(f'BIC: {bic:.4f}\n\n')
    f.write('Coefficients Summary (Sorted by N):\n')
    f.write('-----------------------------------\n')
    f.write(merged_summary_sorted.to_string(index=False))

# Display the sorted merged summary with coefficients and number of observations
merged_summary_sorted

/var/folders/t1/sdbl9_cd061431ynt1cns5hh0000gn/T/ipykernel_4412/980683188.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[sequence_label] = (
/var/folders/t1/sdbl9_cd061431ynt1cns5hh0000gn/T/ipykernel_4412/980683188.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[sequence_label] = (
/var/folders/t1/sdbl9_cd061431ynt1cns5hh0000gn/T/ipykernel_4412/980683188.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer]

,index,Coef.,Std.Err.,t,P>|t|,[0.025,0.975],Significance,Sequence,N
1,low-returns_fast-reframe->low-returns_fast-ref...,5.908843,2.635249,2.242234,0.029,0.641056,11.176631,*,low-returns_fast-reframe->low-returns_fast-ref...,31.0
6,low-returns_fast-reframe->low-returns_slow-ref...,0.146380,3.566420,0.041044,0.967,-6.982791,7.275551,,low-returns_fast-reframe->low-returns_slow-ref...,11.0
10,low-returns_fast-reframe->low-returns_fast-ref...,-4.338468,4.492047,-0.965811,0.338,-13.317942,4.641005,,low-returns_fast-reframe->low-returns_fast-ref...,6.0
4,low-returns_fast-reframe->high-returns_fast-re...,-3.071802,4.843604,-0.634197,0.528,-12.754029,6.610425,,low-returns_fast-reframe->high-returns_fast-re...,5.0
3,low-returns_fast-reframe->low-returns_slow-ref...,3.078198,5.327620,0.577781,0.566,-7.571562,13.727958,,low-returns_fast-reframe->low-returns_slow-ref...,4.0
7,high-returns_fast-reframe->low-returns_fast-re...,0.828198,5.327620,0.155454,0.877,-9.821562,11.477958,,high-returns_fast-reframe->low-returns_fast-re...,4.0
12,low-returns_fast-reframe->low-returns_fast-ref...,-2.671802,5.327620,-0.501500,0.618,-13.321562,7.977958,,low-returns_fast-reframe->low-returns_fast-ref...,4.0
11,low-returns_slow-reframe->low-returns_fast-ref...,-3.005135,6.048849,-0.496811,0.621,-15.096613,9.086343,,low-returns_slow-reframe->low-returns_fast-ref...,3.0
2,low-returns_fast-reframe->low-returns_slow-ref...,7.328198,7.280019,1.006618,0.318,-7.224352,21.880749,,low-returns_fast-reframe->low-returns_slow-ref...,2.0
8,low-returns_slow-reframe->low-returns_slow-ref...,1.828198,7.280019,0.251125,0.803,-12.724352,16.380749,,low-returns_slow-reframe->low-returns_slow-ref...,2.0


In [2]:
mean_levels_complete

37.31645569620253

In [3]:
data_no_outliers

,ID,lvl3,lvl2,lvl1,len_levels_complete,low-returns_fast-reframe->low-returns_fast-reframe->low-returns_fast-reframe,low-returns_fast-reframe->low-returns_slow-reframe->low-returns_slow-reframe,low-returns_fast-reframe->low-returns_slow-reframe->high-returns_fast-reframe,low-returns_fast-reframe->high-returns_fast-reframe->low-returns_fast-reframe,low-returns_slow-reframe->low-returns_fast-reframe->high-returns_fast-reframe,...,high-returns_fast-reframe->low-returns_fast-reframe->low-returns_fast-reframe,low-returns_slow-reframe->low-returns_slow-reframe->low-returns_fast-reframe,low-returns_fast-reframe->high-returns_fast-reframe->high-returns_fast-reframe,low-returns_fast-reframe->low-returns_fast-reframe->high-returns_fast-reframe,low-returns_slow-reframe->low-returns_fast-reframe->low-returns_fast-reframe,low-returns_fast-reframe->low-returns_fast-reframe->low-returns_slow-reframe,low-returns_slow-reframe->low-returns_slow-reframe->high-returns_fast-reframe,high-returns_fast-reframe->low-returns_fast-reframe->high-returns_fast-reframe,high-returns_fast-reframe->high-returns_fast-reframe->low-returns_fast-reframe,low-returns_slow-reframe->low-returns_fast-reframe->low-returns_slow-reframe
1,230,low-returns_fast-reframe,low-returns_fast-reframe,low-returns_fast-reframe,73,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,150,low-returns_fast-reframe,low-returns_fast-reframe,low-returns_fast-reframe,67,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,197,low-returns_fast-reframe,low-returns_fast-reframe,low-returns_fast-reframe,63,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,236,low-returns_slow-reframe,low-returns_slow-reframe,low-returns_fast-reframe,57,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,226,low-returns_fast-reframe,low-returns_fast-reframe,low-returns_fast-reframe,55,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,221,low-returns_fast-reframe,low-returns_fast-reframe,low-returns_fast-reframe,23,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
75,166,low-returns_fast-reframe,low-returns_fast-reframe,low-returns_slow-reframe,22,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
76,248,low-returns_fast-reframe,low-returns_fast-reframe,low-returns_fast-reframe,22,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
77,76,low-returns_fast-reframe,high-returns_fast-reframe,low-returns_fast-reframe,21,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
std_levels_complete

12.113294565293796